# North-Star HF Runner (Kaggle GPU, Legacy Comparison Track)

This notebook is an optional Kaggle-first runner for baseline-vs-variant comparison workflows.

The repository core goals are reliability benchmarking and reproducible evaluation.
This notebook remains for historical compatibility with adapter-based studies.

Typical flow:
1. Authenticate runtime
2. Optionally pull a comparison adapter from Kaggle
3. Run baseline + comparison multi-seed benchmark
4. Compute delta
5. Run transfer matrix and study-gate checks

In [2]:
!nvidia-smi

Thu Apr  2 17:15:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import time
from getpass import getpass
from pathlib import Path

REPO_NAME = 'tool-calling-reliability-benchmark'
REPO_URL = 'https://github.com/aaliyan1230/tool-calling-reliability-benchmark.git'

def find_repo_root(start: Path) -> Path | None:
    for candidate in [start, *start.parents]:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'src').exists():
            return candidate
    return None

repo_root = find_repo_root(Path.cwd())
if repo_root is None:
    kaggle_repo = Path('/kaggle/working') / REPO_NAME
    if not kaggle_repo.exists():
        print(f'[setup] Cloning repo to {kaggle_repo} ...')
        subprocess.run(['git', 'clone', REPO_URL, str(kaggle_repo)], check=True)
    repo_root = kaggle_repo

REPO_ROOT = repo_root.resolve()
os.chdir(REPO_ROOT)

if shutil.which('uv') is None:
    print('[setup] Installing uv ...')
    subprocess.run(['python', '-m', 'pip', 'install', '-q', 'uv'], check=True)

print('Repo root:', REPO_ROOT)
print('Kernel cwd:', Path.cwd())


[setup] Cloning repo to /kaggle/working/tool-calling-reliability-benchmark ...


Cloning into '/kaggle/working/tool-calling-reliability-benchmark'...


Repo root: /kaggle/working/tool-calling-reliability-benchmark
Kernel cwd: /kaggle/working/tool-calling-reliability-benchmark


In [6]:
# Runtime auth (prompt based).
HF_TOKEN = str(os.environ.get('HF_TOKEN', '')).strip()
if not HF_TOKEN:
    HF_TOKEN = getpass('Enter HF_TOKEN (input hidden): ').strip()
if not HF_TOKEN:
    raise RuntimeError('HF_TOKEN is required.')
os.environ['HF_TOKEN'] = HF_TOKEN

KAGGLE_USERNAME = str(os.environ.get('KAGGLE_USERNAME', '')).strip()
if not KAGGLE_USERNAME:
    KAGGLE_USERNAME = input('Enter KAGGLE_USERNAME: ').strip()

KAGGLE_KEY = str(os.environ.get('KAGGLE_KEY', '')).strip()
if not KAGGLE_KEY:
    KAGGLE_KEY = str(os.environ.get('KAGGLE_API_TOKEN', '')).strip()
if not KAGGLE_KEY:
    KAGGLE_KEY = getpass('Enter KAGGLE_KEY (input hidden): ').strip()

if not KAGGLE_USERNAME or not KAGGLE_KEY:
    raise RuntimeError('KAGGLE_USERNAME and KAGGLE_KEY are required.')

os.environ['KAGGLE_USERNAME'] = KAGGLE_USERNAME
os.environ['KAGGLE_KEY'] = KAGGLE_KEY

try:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('[auth] Hugging Face login succeeded.')
except Exception as exc:
    print('[auth] HF login warning:', exc)

print('[auth] Kaggle credentials configured for runtime.')

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


[auth] Hugging Face login succeeded.
[auth] Kaggle credentials configured for runtime.


In [20]:
# Runtime knobs for artifact pull/publish (set automatically).
import os as py_os

py_os.environ['NORTHSTAR_ARTIFACT_DATASET'] = 'aaliyanshaikh/tcrb-qwen25-3b-northstar-artifacts'
py_os.environ['PUBLISH_NORTHSTAR_DATASET_SLUG'] = 'tcrb-qwen25-3b-northstar-artifacts'
py_os.environ['PUBLISH_NORTHSTAR_TITLE'] = 'TCRB Qwen2.5-3B North-Star Artifacts'

print('[runtime] NORTHSTAR_ARTIFACT_DATASET =', py_os.environ['NORTHSTAR_ARTIFACT_DATASET'])
print('[runtime] PUBLISH_NORTHSTAR_DATASET_SLUG =', py_os.environ['PUBLISH_NORTHSTAR_DATASET_SLUG'])

[runtime] NORTHSTAR_ARTIFACT_DATASET = aaliyanshaikh/tcrb-qwen25-3b-northstar-artifacts
[runtime] PUBLISH_NORTHSTAR_DATASET_SLUG = tcrb-qwen25-3b-northstar-artifacts


In [8]:
# Pull latest adapter artifact into local repo tree in the Kaggle runtime.

DATASET = 'aaliyanshaikh/tcrb-qwen25-3b-adapter-artifacts'

# Keep Kaggle clone aligned with current repo so helper scripts exist.
sync_cmd = ['git', 'pull', '--ff-only', 'origin', 'main']
print('Running:', ' '.join(sync_cmd))
sync_res = subprocess.run(sync_cmd, text=True, capture_output=True, check=False)
if sync_res.stdout:
    print(sync_res.stdout)
if sync_res.returncode != 0:
    if sync_res.stderr:
        print(sync_res.stderr)
    raise RuntimeError(f'Git sync failed with code {sync_res.returncode}')

# Some Kaggle images emit sitecustomize warnings when wrapt is absent.
wrapt_install = ['uv', 'pip', 'install', '--python', '.venv/bin/python', 'wrapt']
subprocess.run(wrapt_install, text=True, capture_output=True, check=False)

pull_script = REPO_ROOT / 'scripts' / 'pull_kaggle_adapter.py'
if pull_script.exists():
    pull_cmd = [
        'uv', 'run', 'python', str(pull_script),
        '--dataset', DATASET,
        '--repo-root', '.',
    ]
    print('Running:', ' '.join(pull_cmd))
    res = subprocess.run(pull_cmd, text=True, capture_output=True, check=False)
    if res.stdout:
        print(res.stdout)
    if res.returncode != 0:
        if res.stderr:
            print(res.stderr)
        raise RuntimeError(f'Adapter pull failed with code {res.returncode}')
else:
    print('[adapter] pull_kaggle_adapter.py missing after sync; using direct Kaggle fallback.')
    download_dir = REPO_ROOT / 'tmp' / 'kaggle_adapter_pull'
    if download_dir.exists():
        shutil.rmtree(download_dir)
    download_dir.mkdir(parents=True, exist_ok=True)

    dl_cmd = [
        'kaggle', 'datasets', 'download',
        '-d', DATASET,
        '-p', str(download_dir),
        '--unzip', '-o', '-q',
    ]
    print('Running:', ' '.join(dl_cmd))
    dl_res = subprocess.run(dl_cmd, text=True, capture_output=True, check=False)
    if dl_res.stdout:
        print(dl_res.stdout)
    if dl_res.returncode != 0:
        if dl_res.stderr:
            print(dl_res.stderr)
        raise RuntimeError(f'Kaggle direct download failed with code {dl_res.returncode}')

    src = download_dir / 'adapter'
    dst = REPO_ROOT / 'outputs' / 'ft-notebook' / 'final'
    if not src.exists():
        raise FileNotFoundError(f'Missing adapter dir in downloaded payload: {src}')
    if dst.exists():
        shutil.rmtree(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src, dst)
    print('[adapter] Copied adapter payload to:', dst)

adapter_cfg = REPO_ROOT / 'outputs' / 'ft-notebook' / 'final' / 'adapter_config.json'
if not adapter_cfg.exists():
    raise FileNotFoundError(f'Missing adapter config: {adapter_cfg}')

cfg = json.loads(adapter_cfg.read_text(encoding='utf-8'))
print('[adapter] base_model_name_or_path =', cfg.get('base_model_name_or_path'))
print('[adapter] target_modules count =', len(cfg.get('target_modules', [])))

# Optional: pull already-computed north-star run artifacts from Kaggle dataset.
LABEL_PREFIX = str(os.environ.get('LABEL_PREFIX', 'northstar-hf-kaggle-qwen25-3b')).strip() or 'northstar-hf-kaggle-qwen25-3b'
NORTHSTAR_ARTIFACT_DATASET = str(os.environ.get('NORTHSTAR_ARTIFACT_DATASET', '')).strip()
required = [
    REPO_ROOT / 'runs' / f'{LABEL_PREFIX}-base-ms' / 'multi_seed.json',
    REPO_ROOT / 'runs' / f'{LABEL_PREFIX}-ft-ms' / 'multi_seed.json',
    REPO_ROOT / 'runs' / f'{LABEL_PREFIX}-delta' / 'delta-ms.json',
    REPO_ROOT / 'runs' / f'{LABEL_PREFIX}-matrix' / 'matrix.json',
]

if all(p.exists() for p in required):
    print('[artifacts] Local north-star artifacts already present for', LABEL_PREFIX)
elif NORTHSTAR_ARTIFACT_DATASET:
    pull_runs_script = REPO_ROOT / 'scripts' / 'pull_kaggle_northstar_artifacts.py'
    if not pull_runs_script.exists():
        print('[artifacts] Helper missing in Kaggle clone, continuing without pull:', pull_runs_script)
    else:
        pull_runs_cmd = [
            'uv', 'run', 'python', str(pull_runs_script),
            '--dataset', NORTHSTAR_ARTIFACT_DATASET,
            '--label-prefix', LABEL_PREFIX,
            '--repo-root', '.',
        ]
        print('Running:', ' '.join(pull_runs_cmd))
        pull_runs_res = subprocess.run(pull_runs_cmd, text=True, capture_output=True, check=False)
        if pull_runs_res.stdout:
            print(pull_runs_res.stdout)
        if pull_runs_res.returncode != 0:
            if pull_runs_res.stderr:
                print(pull_runs_res.stderr)
            print('[artifacts] Pull did not succeed; continuing with rerun path.')
else:
    print('[artifacts] Missing local north-star artifacts and NORTHSTAR_ARTIFACT_DATASET is not set.')
    print('[artifacts] To fetch existing runs, set env var NORTHSTAR_ARTIFACT_DATASET=owner/slug and rerun this cell.')

Running: git pull --ff-only origin main
Already up to date.

Running: uv run python /kaggle/working/tool-calling-reliability-benchmark/scripts/pull_kaggle_adapter.py --dataset aaliyanshaikh/tcrb-qwen25-3b-adapter-artifacts --repo-root .
Dataset URL: https://www.kaggle.com/datasets/aaliyanshaikh/tcrb-qwen25-3b-adapter-artifacts
License(s): CC0-1.0

{
  "dataset": "aaliyanshaikh/tcrb-qwen25-3b-adapter-artifacts",
  "download_dir": "/kaggle/working/tool-calling-reliability-benchmark/tmp/kaggle_adapter_pull",
  "adapter_root": "/kaggle/working/tool-calling-reliability-benchmark/tmp/kaggle_adapter_pull/adapter",
  "target_dir": "/kaggle/working/tool-calling-reliability-benchmark/outputs/ft-notebook/final"
}
Adapter materialized successfully.

[adapter] base_model_name_or_path = Qwen/Qwen2.5-3B-Instruct
[adapter] target_modules count = 7
[artifacts] Helper missing in Kaggle clone, continuing without pull: /kaggle/working/tool-calling-reliability-benchmark/scripts/pull_kaggle_northstar_artifa

In [9]:
# Ensure benchmark/runtime dependencies exist in the repo's uv environment.
required_mods = ['torch', 'transformers', 'peft', 'trl', 'datasets', 'accelerate', 'bitsandbytes']

# Verify modules in uv-managed environment (not only notebook kernel env).
probe = [
    'uv', 'run', 'python', '-c',
    "import importlib.util as u; mods=%r; missing=[m for m in mods if u.find_spec(m) is None]; print('MISSING=' + ','.join(missing))" % required_mods,
 ]
probe_res = subprocess.run(probe, text=True, capture_output=True, check=False)
probe_out = (probe_res.stdout or '').strip()
print('[deps] Probe output:', probe_out)

missing = []
if 'MISSING=' in probe_out:
    missing_text = probe_out.split('MISSING=', 1)[1].strip()
    if missing_text:
        missing = [m for m in missing_text.split(',') if m]

if missing:
    # Install directly into uv venv interpreter used by `uv run`.
    install_cmd = [
        'uv', 'pip', 'install', '--python', '.venv/bin/python',
        'torch', 'transformers', 'peft', 'trl', 'datasets', 'accelerate', 'bitsandbytes', 'wrapt',
    ]
    print('Running:', ' '.join(install_cmd))
    install_res = subprocess.run(install_cmd, text=True, capture_output=True, check=False)
    if install_res.stdout:
        print(install_res.stdout[-4000:])
    if install_res.returncode != 0:
        if install_res.stderr:
            print(install_res.stderr[-4000:])
        raise RuntimeError(f'uv pip install failed with code {install_res.returncode}')

    recheck = subprocess.run(probe, text=True, capture_output=True, check=False)
    recheck_out = (recheck.stdout or '').strip()
    print('[deps] Recheck output:', recheck_out)
    if 'MISSING=' in recheck_out and recheck_out.split('MISSING=', 1)[1].strip():
        raise RuntimeError(f'Modules still missing in uv env: {recheck_out}')
    print('[deps] uv environment dependencies are ready.')
else:
    print('[deps] uv environment already has required modules.')

[deps] Probe output: MISSING=torch,transformers,peft,trl,datasets,accelerate,bitsandbytes
Running: uv pip install --python .venv/bin/python torch transformers peft trl datasets accelerate bitsandbytes wrapt
[deps] Recheck output: MISSING=
[deps] uv environment dependencies are ready.


In [10]:
# Run full HF north-star pipeline (base ms, ft ms, delta, matrix).
LABEL_PREFIX = 'northstar-hf-kaggle-qwen25-3b'

# Matrix gate thresholds. Set target sequence to 0.0 for matrix-not-fail calibration runs.
MATRIX_TARGET_FIRST_MIN_DELTA = 0.03
MATRIX_TARGET_SEQ_MIN_DELTA = 0.00
MATRIX_OPEN_FIRST_MIN_DELTA = -0.03
MATRIX_OPEN_SEQ_MIN_DELTA = -0.03

# Optional study-gate stage after matrix generation.
RUN_STUDY_GATE = True
STUDY_GATE_REQUIRE_MATRIX_NOT_FAIL = True
STUDY_GATE_REQUIRE_MATRIX_SIGNAL = False
STUDY_GATE_FAIL_ON_VIOLATION = False

CMD = [
    'uv', 'run', 'python', 'scripts/run_northstar_hf.py',
    '--base-planner-config', 'configs/planners/hf_qwen2_5_3b_base.json',
    '--ft-planner-config', 'configs/planners/hf_qwen2_5_3b_ft.json',
    '--label-prefix', LABEL_PREFIX,
    '--matrix-target-first-min-delta', str(MATRIX_TARGET_FIRST_MIN_DELTA),
    '--matrix-target-seq-min-delta', str(MATRIX_TARGET_SEQ_MIN_DELTA),
    '--matrix-open-first-min-delta', str(MATRIX_OPEN_FIRST_MIN_DELTA),
    '--matrix-open-seq-min-delta', str(MATRIX_OPEN_SEQ_MIN_DELTA),
]

if RUN_STUDY_GATE:
    CMD.append('--run-study-gate')
if STUDY_GATE_REQUIRE_MATRIX_NOT_FAIL:
    CMD.append('--study-gate-require-matrix-not-fail')
if STUDY_GATE_REQUIRE_MATRIX_SIGNAL:
    CMD.append('--study-gate-require-matrix-signal')
if STUDY_GATE_FAIL_ON_VIOLATION:
    CMD.append('--study-gate-fail-on-violation')

print('Running:', ' '.join(CMD))
started = time.time()
proc = subprocess.Popen(CMD, text=True, cwd=str(REPO_ROOT), stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end='')
rc = proc.wait()
elapsed = time.time() - started
print(f'[northstar] Total elapsed: {elapsed:.1f}s')
if rc != 0:
    raise RuntimeError(f'North-star run failed with code {rc}')

Running: uv run python scripts/run_northstar_hf.py --base-planner-config configs/planners/hf_qwen2_5_3b_base.json --ft-planner-config configs/planners/hf_qwen2_5_3b_ft.json --label-prefix northstar-hf-kaggle-qwen25-3b
[northstar] HF_TOKEN set: True
[northstar] Running: uv run tcrb multi-seed --config configs/baseline.json --workload workloads/sample_tasks.json --seeds 11,22,33 --planner-config configs/planners/hf_qwen2_5_3b_base.json --label northstar-hf-kaggle-qwen25-3b-base-ms

Fetching 2 files: 100%|██████████| 2/2 [00:32<00:00, 16.19s/it]

Loading weights: 100%|██████████| 434/434 [00:07<00:00, 57.36it/s]
Planner: hf_qwen2_5_3b_base
Wrote multi-seed results: runs/northstar-hf-kaggle-qwen25-3b-base-ms/multi_seed.json
Wrote multi-seed summary: runs/northstar-hf-kaggle-qwen25-3b-base-ms/multi_seed_summary.md
[northstar] Stage finished in 63.9s
[northstar] Running: uv run tcrb multi-seed --config configs/baseline.json --workload workloads/sample_tasks.json --seeds 11,22,33 --planner-co

In [17]:
# Optional: publish fresh north-star artifacts to Kaggle dataset for later reuse.
import os as py_os

PUBLISH_NORTHSTAR_DATASET_SLUG = str(py_os.environ.get('PUBLISH_NORTHSTAR_DATASET_SLUG', '')).strip()
PUBLISH_NORTHSTAR_TITLE = str(py_os.environ.get('PUBLISH_NORTHSTAR_TITLE', 'TCRB Qwen2.5-3B North-Star Artifacts')).strip()

if PUBLISH_NORTHSTAR_DATASET_SLUG:
    publish_script = REPO_ROOT / 'scripts' / 'publish_kaggle_northstar_artifacts.py'
    if publish_script.exists():
        publish_cmd = [
            'uv', 'run', 'python', str(publish_script),
            '--dataset-slug', PUBLISH_NORTHSTAR_DATASET_SLUG,
            '--title', PUBLISH_NORTHSTAR_TITLE,
            '--label-prefix', LABEL_PREFIX,
            '--repo-root', '.',
        ]
        print('Running:', ' '.join(publish_cmd))
        publish_res = subprocess.run(publish_cmd, text=True, capture_output=True, check=False)
        if publish_res.stdout:
            print(publish_res.stdout)
        if publish_res.returncode != 0:
            if publish_res.stderr:
                print(publish_res.stderr)
            raise RuntimeError(f'Artifact publish failed with code {publish_res.returncode}')
    else:
        print('[publish] Helper missing in Kaggle clone; using inline publish fallback.')
        stage_dir = REPO_ROOT / 'tmp' / 'kaggle_northstar_publish_inline'
        if stage_dir.exists():
            shutil.rmtree(stage_dir)
        stage_runs = stage_dir / 'runs'
        stage_runs.mkdir(parents=True, exist_ok=True)

        required = [f'{LABEL_PREFIX}-base-ms', f'{LABEL_PREFIX}-ft-ms', f'{LABEL_PREFIX}-delta', f'{LABEL_PREFIX}-matrix']
        for name in required:
            src = REPO_ROOT / 'runs' / name
            dst = stage_runs / name
            if not src.exists():
                raise FileNotFoundError(f'Missing run directory for publish: {src}')
            shutil.copytree(src, dst)

        staged_files = [p for p in stage_dir.rglob('*') if p.is_file()]
        print('[publish] staged file count =', len(staged_files))
        if not staged_files:
            raise RuntimeError('No files staged for Kaggle publish.')

        dataset_id = f"{py_os.environ['KAGGLE_USERNAME']}/{PUBLISH_NORTHSTAR_DATASET_SLUG}"
        metadata = {
            'title': PUBLISH_NORTHSTAR_TITLE,
            'id': dataset_id,
            'licenses': [{'name': 'CC0-1.0'}],
        }
        (stage_dir / 'dataset-metadata.json').write_text(json.dumps(metadata, indent=2) + '\n', encoding='utf-8')

        create_cmd = ['kaggle', 'datasets', 'create', '-p', str(stage_dir), '-r', 'zip', '-q']
        create_res = subprocess.run(create_cmd, text=True, capture_output=True, check=False)
        create_text = (create_res.stdout or '') + '\n' + (create_res.stderr or '')
        if create_res.returncode == 0 and 'error' not in create_text.lower():
            if create_res.stdout:
                print(create_res.stdout)
            print('[publish] Created Kaggle dataset:', dataset_id)
        else:
            if create_res.stdout:
                print(create_res.stdout)
            if create_res.stderr:
                print(create_res.stderr)

            if 'already exists' in create_text.lower():
                version_cmd = [
                    'kaggle', 'datasets', 'version', '-p', str(stage_dir),
                    '-m', f'Update north-star artifacts for {LABEL_PREFIX}',
                    '-r', 'zip', '-q'
                ]
                version_res = subprocess.run(version_cmd, text=True, capture_output=True, check=False)
                version_text = (version_res.stdout or '') + '\n' + (version_res.stderr or '')
                if version_res.returncode != 0 or 'error' in version_text.lower():
                    raise RuntimeError('Inline dataset version failed.')
                if version_res.stdout:
                    print(version_res.stdout)
                print('[publish] Updated Kaggle dataset version:', dataset_id)
            else:
                raise RuntimeError('Inline dataset create failed.')
else:
    print('[publish] Skipped. Set PUBLISH_NORTHSTAR_DATASET_SLUG to publish runs/ artifacts to Kaggle.')

[publish] Helper missing in Kaggle clone; using inline publish fallback.
[publish] staged file count = 26
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/aaliyanshaikh/tcrb-qwen25-3b-northstar-artifacts

[publish] Created Kaggle dataset: aaliyanshaikh/tcrb-qwen25-3b-northstar-artifacts


## Artifact Index

This section only lists generated artifact locations for quick navigation and reporting.

In [12]:
label = 'northstar-hf-kaggle-qwen25-3b'
paths = [
    REPO_ROOT / 'runs' / f'{label}-base-ms' / 'multi_seed.json',
    REPO_ROOT / 'runs' / f'{label}-ft-ms' / 'multi_seed.json',
    REPO_ROOT / 'runs' / f'{label}-delta' / 'delta-ms.json',
    REPO_ROOT / 'runs' / f'{label}-matrix' / 'matrix.json',
]

print('Artifact files:')
for p in paths:
    print('-', p, 'exists=' + str(p.exists()))

Artifact files:
- /kaggle/working/tool-calling-reliability-benchmark/runs/northstar-hf-kaggle-qwen25-3b-base-ms/multi_seed.json exists=True
- /kaggle/working/tool-calling-reliability-benchmark/runs/northstar-hf-kaggle-qwen25-3b-ft-ms/multi_seed.json exists=True
- /kaggle/working/tool-calling-reliability-benchmark/runs/northstar-hf-kaggle-qwen25-3b-delta/delta-ms.json exists=True
- /kaggle/working/tool-calling-reliability-benchmark/runs/northstar-hf-kaggle-qwen25-3b-matrix/matrix.json exists=True


In [13]:
# Final study findings (natural-language summary with concrete numbers)

from pathlib import Path

import json

from statistics import mean



label = globals().get("LABEL_PREFIX", "northstar-hf-kaggle-qwen25-3b")

root = Path("runs")



base_ms_path = root / f"{label}-base-ms" / "multi_seed.json"

ft_ms_path = root / f"{label}-ft-ms" / "multi_seed.json"

delta_path = root / f"{label}-delta" / "delta-ms.json"

matrix_path = root / f"{label}-matrix" / "matrix.json"



def load_json(path: Path):

    if not path.exists():

        return None

    return json.loads(path.read_text(encoding="utf-8"))



base_ms = load_json(base_ms_path)

ft_ms = load_json(ft_ms_path)

delta = load_json(delta_path)

matrix = load_json(matrix_path)



lines = []

lines.append("## Final Study Findings")

lines.append("")

lines.append("This summary interprets the latest north-star artifacts in plain language.")

lines.append("")



if base_ms and ft_ms:

    b = {r.get("policy"): r.get("metrics", {}) for r in base_ms.get("aggregate_policy_metrics", [])}

    f = {r.get("policy"): r.get("metrics", {}) for r in ft_ms.get("aggregate_policy_metrics", [])}

    policies = sorted(set(b.keys()) & set(f.keys()))



    success_deltas = []

    invalid_deltas = []

    for p in policies:

        bs = b[p].get("task_success_rate", {}).get("mean")

        fs = f[p].get("task_success_rate", {}).get("mean")

        bi = b[p].get("invalid_tool_call_rate", {}).get("mean")

        fi = f[p].get("invalid_tool_call_rate", {}).get("mean")

        if bs is not None and fs is not None:

            success_deltas.append(float(fs) - float(bs))

        if bi is not None and fi is not None:

            invalid_deltas.append(float(fi) - float(bi))



    if success_deltas:

        lines.append(f"Across {len(policies)} policies, mean success-rate delta (FT - Base) is {mean(success_deltas):+.4f}.")

    if invalid_deltas:

        lines.append(f"Mean invalid-tool-call-rate delta is {mean(invalid_deltas):+.4f} (negative is better).")

else:

    lines.append("Base/finetuned multi-seed artifacts were not both found, so aggregate deltas could not be computed.")



if delta:

    target_rows = delta.get("target", {}).get("policies", [])

    if target_rows:

        t_success = [float(r.get("delta", {}).get("task_success_rate")) for r in target_rows if r.get("delta", {}).get("task_success_rate") is not None]

        t_invalid = [float(r.get("delta", {}).get("invalid_tool_call_rate")) for r in target_rows if r.get("delta", {}).get("invalid_tool_call_rate") is not None]

        if t_success:

            lines.append(f"On target policies, average success-rate delta is {mean(t_success):+.4f}.")

        if t_invalid:

            lines.append(f"On target policies, average invalid-tool-call-rate delta is {mean(t_invalid):+.4f}.")



if matrix and matrix.get("rows"):

    rows = matrix.get("rows", [])

    target_rows = [r for r in rows if str(r.get("split", "")) == "target"]

    open_rows = [r for r in rows if str(r.get("split", "")) == "open"]



    if target_rows:

        tf = [float(r.get("delta_first_tool_accuracy", 0.0)) for r in target_rows]

        ts = [float(r.get("delta_sequence_prefix_accuracy", 0.0)) for r in target_rows]

        lines.append(f"Transfer on target toolsets: first-tool delta {mean(tf):+.4f}, sequence delta {mean(ts):+.4f}.")



    if open_rows:

        of = [float(r.get("delta_first_tool_accuracy", 0.0)) for r in open_rows]

        os = [float(r.get("delta_sequence_prefix_accuracy", 0.0)) for r in open_rows]

        lines.append(f"Transfer on open toolsets: first-tool delta {mean(of):+.4f}, sequence delta {mean(os):+.4f}.")



    verdict = matrix.get("portfolio_verdict")

    if verdict:

        lines.append(f"Overall transfer-matrix verdict: {verdict}.")



lines.append("")

lines.append("Interpretation:")

lines.append("- Positive success-rate deltas mean the finetuned planner solved more tasks than base.")

lines.append("- Negative invalid-call deltas mean fewer malformed/incorrect tool calls.")

lines.append("- Better target with worse open performance suggests specialization with weaker generalization.")

lines.append("")

lines.append(f"Source base artifact: {base_ms_path}")

lines.append(f"Source finetuned artifact: {ft_ms_path}")

lines.append(f"Source delta artifact: {delta_path}")

lines.append(f"Source matrix artifact: {matrix_path}")



print("\n".join(lines))


## Final Study Findings

This summary interprets the latest north-star artifacts in plain language.

Across 4 policies, mean success-rate delta (FT - Base) is +0.0000.
Mean invalid-tool-call-rate delta is +0.0000 (negative is better).
On target policies, average success-rate delta is +0.0000.
On target policies, average invalid-tool-call-rate delta is +0.0000.
Transfer on target toolsets: first-tool delta +0.0000, sequence delta +0.0000.
Transfer on open toolsets: first-tool delta +0.0000, sequence delta +0.0000.
Overall transfer-matrix verdict: FAIL.

Interpretation:
- Positive success-rate deltas mean the finetuned planner solved more tasks than base.
- Negative invalid-call deltas mean fewer malformed/incorrect tool calls.
- Better target with worse open performance suggests specialization with weaker generalization.

Source base artifact: runs/northstar-hf-kaggle-qwen25-3b-base-ms/multi_seed.json
Source finetuned artifact: runs/northstar-hf-kaggle-qwen25-3b-ft-ms/multi_seed.json
So

In [18]:
# Verify Kaggle dataset file visibility from runtime.
datasets_to_check = [
    'aaliyanshaikh/tcrb-qwen25-3b-adapter-artifacts',
    'aaliyanshaikh/tcrb-qwen25-3b-northstar-artifacts',
]

for ds in datasets_to_check:
    print('\n[dataset]', ds)
    cmd = ['kaggle', 'datasets', 'files', ds]
    res = subprocess.run(cmd, text=True, capture_output=True, check=False)
    if res.stdout:
        print(res.stdout)
    if res.returncode != 0:
        if res.stderr:
            print(res.stderr)
        print('[dataset] listing failed with code', res.returncode)

print('\n[dataset] Search check for northstar slug:')
search_cmd = ['kaggle', 'datasets', 'list', '--search', 'tcrb-qwen25-3b-northstar-artifacts']
search_res = subprocess.run(search_cmd, text=True, capture_output=True, check=False)
if search_res.stdout:
    print(search_res.stdout)
if search_res.returncode != 0 and search_res.stderr:
    print(search_res.stderr)


[dataset] aaliyanshaikh/tcrb-qwen25-3b-adapter-artifacts
name                                   size  creationDate                
---------------------------------  --------  --------------------------  
adapter/README.md                      5214  2026-04-02 14:05:06.545000  
adapter/adapter_config.json            1055  2026-04-02 14:05:06.745000  
adapter/adapter_model.safetensors  59934640  2026-04-02 14:05:09.203000  
adapter/chat_template.jinja            2507  2026-04-02 14:05:08.026000  
adapter/tokenizer.json             11421892  2026-04-02 14:05:06.946000  
adapter/tokenizer_config.json           665  2026-04-02 14:05:08.036000  
adapter/training_args.bin              5649  2026-04-02 14:05:06.718000  
published_at_utc.txt                     21  2026-04-02 14:05:05.809000  


[dataset] aaliyanshaikh/tcrb-qwen25-3b-northstar-artifacts
Next Page Token = CfDJ8OuP2e3tnwRHgoiuJLZ8zbzFljHurACd2dllD6rva-y41rNf4LsORVFT9ae0bVaELvi1Kp-yjHN7YXFp1fb9BFIPJMKz81BajO4PK7bkwxIhW3lU0_ZpC3W